In [ ]:
# 데이터 준비
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs, make_classification, make_regression
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

# Helper function to create DataFrame
def create_classification_data():
    X, y = make_classification(n_samples=100, n_features=5, random_state=42)
    df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
    df['target'] = y
    
    # feature_1에 결측치를 추가 (10% 비율로)
    missing_indices = np.random.choice(df.index, size=int(len(df) * 0.1), replace=False)
    df.loc[missing_indices, 'feature_1'] = np.nan
    
    return df

def create_regression_data():
    X, y = make_regression(n_samples=100, n_features=5, noise=0.1, random_state=42)
    df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
    df['target'] = y
    
    for column in df.columns[:-1]:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # 이상치를 추가할 인덱스를 랜덤하게 선택
        outlier_indices = np.random.choice(df.index, size=5, replace=False)
        for idx in outlier_indices:
            df.at[idx, column] = np.random.uniform(upper_bound + 1, upper_bound + 10)

    return df

def create_blobs_data():
    X, y = make_blobs(n_samples=100, n_features=5, centers=3, random_state=42)
    df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
    df['target'] = y
    
    return df


# Generate datasets for each problem type
classification_df = create_classification_data()
regression_df = create_regression_data()
blobs_df = create_blobs_data()

print('=== Classification Generated Datasets ===')
print(classification_df.head())
print('\n=== regression Generated Datasets ===')
print(regression_df.head())
print('\n=== blobs Generated Datasets ===')
print(blobs_df.head())

In [ ]:
print('[전처리 전] 결측치 수:', classification_df.isna().sum().sum())

classification_df['feature_1'] = classification_df['feature_1'].fillna(classification_df['feature_1'].mean())

print('[전처리 후] 결측치 수:', classification_df.isna().sum().sum())

In [ ]:
print('이상치 처리 전 결측치 수:', regression_df['feature_3'].isna().sum())

Q1 = regression_df['feature_3'].quantile(0.25)
Q3 = regression_df['feature_3'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

regression_df.loc[(regression_df['feature_3'] < lower_bound) | (regression_df['feature_3'] > upper_bound), 'feature_3'] = None

print('이상치 처리 후 결측치 수:', regression_df['feature_3'].isna().sum())

regression_df.dropna(subset=['feature_3'], inplace=True)

print('결측치 삭제:', regression_df['feature_3'].isna().sum())